Import bibliotek

In [1]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

Funkcje

In [2]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(name)

    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, y_pred), 4))
    print("F1:", round(f1_score(y_test, y_pred), 4))
    print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
    print("PR-AUC:", round(average_precision_score(y_test, y_proba), 4))

    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    
    print("Classification Report:\n", classification_report(y_test, y_pred, zero_division=0))


def get_metrics(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba)
    }

def get_false_negatives(model):
    return debug_df[(debug_df['y_true']==1) & (debug_df[f'{model}_pred']== 0)]

Wczytanie danych

In [3]:
X_train = pd.read_csv('../data/processed/X_train_scaled.csv')
X_test = pd.read_csv('../data/processed/X_test_scaled.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

Test wczytania danych

In [4]:
display(X_train.shape)
display(X_test.shape)
display(y_train.shape)
display(y_test.shape)

display(X_train.head())
display(y_train.value_counts())
display(y_test.value_counts())

(8000, 8)

(2000, 8)

(8000,)

(2000,)

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min]
0,1,0,0.874402,-0.800498,-0.100148,-0.851496,-1.182391,1.485820
1,1,0,-0.408876,1.900381,-0.122496,-0.329786,-0.421646,1.973739
2,0,1,0.536697,-0.500400,-0.167192,0.011332,0.052397,-1.300041
3,0,1,-0.071171,1.100121,-0.692367,1.325640,1.499616,1.359905
4,1,0,0.198993,-0.900531,-0.223061,-0.189326,-0.280724,-0.733425


Machine failure
0    7736
1     264
Name: count, dtype: int64

Machine failure
0    1934
1      66
Name: count, dtype: int64

Tworzenie modelu naiwnego

In [5]:
dummy = DummyClassifier(strategy='most_frequent')

dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_proba_dummy = dummy.predict_proba(X_test)[:, 1]

In [6]:
evaluate_model('Dummy Classifier', dummy, X_test, y_test)

Dummy Classifier
Accuracy: 0.967
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.5
PR-AUC: 0.033
Confusion Matrix:
 [[1934    0]
 [  66    0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98      1934
           1       0.00      0.00      0.00        66

    accuracy                           0.97      2000
   macro avg       0.48      0.50      0.49      2000
weighted avg       0.94      0.97      0.95      2000



Wnioski:



Logistic Regression

In [7]:
log_reg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)
y_proba_log = log_reg.predict_proba(X_test)[:, 1]


In [8]:
evaluate_model('LogisticRegression', log_reg, X_test, y_test)

LogisticRegression
Accuracy: 0.8475
Precision: 0.1595
Recall: 0.8485
F1: 0.2686
ROC-AUC: 0.9202
PR-AUC: 0.4192
Confusion Matrix:
 [[1639  295]
 [  10   56]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.85      0.91      1934
           1       0.16      0.85      0.27        66

    accuracy                           0.85      2000
   macro avg       0.58      0.85      0.59      2000
weighted avg       0.97      0.85      0.89      2000



Wnioski:



Random Forest

In [9]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

random_forest.fit(X_train, y_train)

evaluate_model('RandomForestClassifier', random_forest, X_test, y_test)

RandomForestClassifier
Accuracy: 0.99
Precision: 0.8594
Recall: 0.8333
F1: 0.8462
ROC-AUC: 0.991
PR-AUC: 0.893
Confusion Matrix:
 [[1925    9]
 [  11   55]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99      1934
           1       0.86      0.83      0.85        66

    accuracy                           0.99      2000
   macro avg       0.93      0.91      0.92      2000
weighted avg       0.99      0.99      0.99      2000



Gradient Boosting


In [10]:
gradient_boosting = GradientBoostingClassifier(
    random_state=42
)

gradient_boosting.fit(X_train, y_train)

evaluate_model('GradientBoostingClassifier', gradient_boosting, X_test, y_test)

GradientBoostingClassifier
Accuracy: 0.9895
Precision: 0.8814
Recall: 0.7879
F1: 0.832
ROC-AUC: 0.9943
PR-AUC: 0.9145
Confusion Matrix:
 [[1927    7]
 [  14   52]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99      1934
           1       0.88      0.79      0.83        66

    accuracy                           0.99      2000
   macro avg       0.94      0.89      0.91      2000
weighted avg       0.99      0.99      0.99      2000



Porównanie modeli

In [11]:
results = []

results.append(get_metrics("Dummy", dummy, X_test, y_test))
results.append(get_metrics("Logistic Regression", log_reg, X_test, y_test))
results.append(get_metrics("Random Forest", random_forest, X_test, y_test))
results.append(get_metrics("Gradient Boosting", gradient_boosting, X_test, y_test))

results_df = pd.DataFrame(results)
display(results_df.round(4))

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Dummy,0.9670,0.0000,0.0000,0.0000,0.5000,0.0330
1,Logistic Regression,0.8475,0.1595,0.8485,0.2686,0.9202,0.4192
2,Random Forest,0.9900,0.8594,0.8333,0.8462,0.9910,0.8930
3,Gradient Boosting,0.9895,0.8814,0.7879,0.8320,0.9943,0.9145


In [12]:
#results_df.to_csv("../data/processed/model_results_baseline.csv", index=False)

## Analiza przeoczonych awarii - False Negatives

In [13]:
from sklearn.model_selection import train_test_split

df_clean = pd.read_csv('../data/processed/produkcja_clean.csv')

X_raw = df_clean.drop(columns=['Machine failure'])
y_raw = df_clean['Machine failure']

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw,
    y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw
)

X_test_raw = X_test_raw.copy()

In [14]:
print("X_test scaled shape:", X_test.shape)
print("X_test raw shape:", X_test_raw.shape)

print("y_test shape:", y_test.shape)
print("y_test_raw shape:", y_test_raw.shape)

X_test scaled shape: (2000, 8)
X_test raw shape: (2000, 8)
y_test shape: (2000,)
y_test_raw shape: (2000,)


In [15]:
print((y_test.reset_index(drop=True) == y_test_raw.reset_index(drop=True)).all())

True


In [16]:
debug_df = X_test_raw.copy()
debug_df['y_true'] = y_test_raw.values

models = {
    'log_reg': log_reg,
    'random_forest': random_forest,
    'gradient_boosting': gradient_boosting
}

for name, model in models.items():
    debug_df[f"{name}_pred"] = model.predict(X_test)
    debug_df[f"{name}_proba"] = model.predict_proba(X_test)[:, 1]

display(debug_df.head())
display(debug_df.shape)

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba
7566,1,0,311.0,10.7,1613,34.7,5861.28,142,0,0,0.041565,0,0.000,0,0.000622
8199,1,0,310.7,11.5,1737,27.0,4911.25,225,1,0,0.070138,0,0.150,0,0.138585
9316,0,0,309.4,10.9,1417,41.2,6113.58,155,0,0,0.086594,0,0.005,0,0.000668
2882,1,0,309.6,9.0,1684,32.8,5784.22,53,0,0,0.032116,0,0.000,0,0.000638
4293,1,0,310.0,8.2,1393,44.9,6549.77,208,0,1,0.879893,0,0.410,0,0.037122


(2000, 15)

In [17]:
debug_df[[
    "y_true",
    "log_reg_pred",
    "log_reg_proba",
    "random_forest_pred",
    "random_forest_proba",
    "gradient_boosting_pred",
    "gradient_boosting_proba"
]].head(10)

,y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba
7566,0,0,0.041565,0,0.000,0,0.000622
8199,1,0,0.070138,0,0.150,0,0.138585
9316,0,0,0.086594,0,0.005,0,0.000668
2882,0,0,0.032116,0,0.000,0,0.000638
4293,0,1,0.879893,0,0.410,0,0.037122
4261,0,0,0.183033,0,0.000,0,0.002261
4485,0,0,0.090754,0,0.000,0,0.002235
2518,0,0,0.115763,0,0.000,0,0.000668
8084,0,0,0.095162,0,0.000,0,0.000622
9661,0,0,0.306636,0,0.085,0,0.054976


In [18]:
get_false_negatives('log_reg')

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba
8199,1,0,310.7,11.5,1737,27.0,4911.25,225,1,0,0.070138,0,0.150,0,0.138585
4151,0,1,310.4,8.5,1373,48.0,6901.45,73,1,0,0.443058,1,0.895,1,0.943080
4475,1,0,310.5,7.8,1351,41.8,5913.71,10,1,0,0.163843,0,0.415,1,0.965152
4034,1,0,310.8,8.8,1615,29.0,4904.55,235,1,0,0.431925,0,0.150,0,0.053293
4833,1,0,311.9,8.5,1377,41.6,5998.68,34,1,0,0.131180,0,0.340,1,0.965545
8609,1,0,308.3,10.9,1475,40.5,6255.70,222,1,0,0.249070,0,0.110,0,0.057842
9018,1,0,308.1,10.8,1615,35.4,5986.93,217,1,0,0.132736,0,0.035,0,0.053958
442,1,0,308.5,11.1,1399,61.5,9009.93,61,1,0,0.399294,1,0.710,1,0.975880
3787,1,0,310.8,8.5,1377,47.3,6820.62,22,1,0,0.248337,1,0.785,1,0.943080
7510,1,0,311.8,11.3,1524,38.9,6208.16,214,1,0,0.137707,0,0.125,0,0.045325


In [19]:
display(get_false_negatives('random_forest'))

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba
8199,1,0,310.7,11.5,1737,27.0,4911.25,225,1,0,0.070138,0,0.150,0,0.138585
4475,1,0,310.5,7.8,1351,41.8,5913.71,10,1,0,0.163843,0,0.415,1,0.965152
4034,1,0,310.8,8.8,1615,29.0,4904.55,235,1,0,0.431925,0,0.150,0,0.053293
4833,1,0,311.9,8.5,1377,41.6,5998.68,34,1,0,0.131180,0,0.340,1,0.965545
2941,0,1,309.6,8.9,1996,19.8,4138.61,203,1,1,0.605943,0,0.070,0,0.015848
8609,1,0,308.3,10.9,1475,40.5,6255.70,222,1,0,0.249070,0,0.110,0,0.057842
8026,1,0,311.9,11.2,1399,50.2,7354.45,222,1,1,0.635897,0,0.320,0,0.319052
9663,1,0,310.1,11.0,1435,48.8,7333.32,229,1,1,0.590242,0,0.345,0,0.126378
9758,1,0,309.8,11.2,2271,16.2,3852.66,218,1,1,0.778214,0,0.080,0,0.091027
9018,1,0,308.1,10.8,1615,35.4,5986.93,217,1,0,0.132736,0,0.035,0,0.053958


In [20]:
display(get_false_negatives('gradient_boosting'))

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba
8199,1,0,310.7,11.5,1737,27.0,4911.25,225,1,0,0.070138,0,0.150,0,0.138585
4034,1,0,310.8,8.8,1615,29.0,4904.55,235,1,0,0.431925,0,0.150,0,0.053293
2494,1,0,308.8,9.7,1329,53.6,7459.65,207,1,1,0.934708,1,0.605,0,0.494805
2941,0,1,309.6,8.9,1996,19.8,4138.61,203,1,1,0.605943,0,0.070,0,0.015848
1085,1,0,307.8,10.8,1385,56.4,8180.08,202,1,1,0.837665,1,0.695,0,0.276324
8609,1,0,308.3,10.9,1475,40.5,6255.70,222,1,0,0.249070,0,0.110,0,0.057842
8026,1,0,311.9,11.2,1399,50.2,7354.45,222,1,1,0.635897,0,0.320,0,0.319052
9653,1,0,309.9,10.9,1373,55.7,8008.56,201,1,1,0.826901,1,0.660,0,0.135555
3760,1,0,310.9,8.6,1377,46.8,6748.52,166,1,1,0.788600,1,0.690,0,0.448550
9663,1,0,310.1,11.0,1435,48.8,7333.32,229,1,1,0.590242,0,0.345,0,0.126378


In [21]:
fn_summary = pd.DataFrame({
    'model': ['log_reg', 'random_forest', 'gradient_boosting'],
    'false_negatives': [get_false_negatives('log_reg').shape[0],
                        get_false_negatives('random_forest').shape[0],
                        get_false_negatives('gradient_boosting').shape[0]]
})
display(fn_summary)

,model,false_negatives
0,log_reg,10
1,random_forest,11
2,gradient_boosting,14


In [22]:
fn_log_index = set(get_false_negatives('log_reg').index)
fn_rf_index = set(get_false_negatives('random_forest').index)
fn_gb_index = set(get_false_negatives('gradient_boosting').index)

print("Logistic Regression FN:", len(fn_log_index))
print("Random Forest FN:", len(fn_rf_index))
print("Gradient Boosting FN:", len(fn_gb_index))

Logistic Regression FN: 10
Random Forest FN: 11
Gradient Boosting FN: 14


In [23]:
fn_all_index = fn_log_index & fn_rf_index & fn_gb_index

print("Awarie przeoczone przez wszystkie modele:", len(fn_all_index))
print(fn_all_index)

Awarie przeoczone przez wszystkie modele: 5
{8609, 4034, 8199, 7510, 9018}


In [24]:
fn_all = debug_df.loc[list(fn_all_index)].copy()
display(fn_all)

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba
8609,1,0,308.3,10.9,1475,40.5,6255.70,222,1,0,0.249070,0,0.110,0,0.057842
4034,1,0,310.8,8.8,1615,29.0,4904.55,235,1,0,0.431925,0,0.150,0,0.053293
8199,1,0,310.7,11.5,1737,27.0,4911.25,225,1,0,0.070138,0,0.150,0,0.138585
7510,1,0,311.8,11.3,1524,38.9,6208.16,214,1,0,0.137707,0,0.125,0,0.045325
9018,1,0,308.1,10.8,1615,35.4,5986.93,217,1,0,0.132736,0,0.035,0,0.053958


In [25]:
overlap_summary = pd.DataFrame({
    "comparison": [
        "Logistic Regression ∩ Random Forest",
        "Logistic Regression ∩ Gradient Boosting",
        "Random Forest ∩ Gradient Boosting",
        "All three models"
    ],
    "shared_false_negatives": [
        len(fn_log_index & fn_rf_index),
        len(fn_log_index & fn_gb_index),
        len(fn_rf_index & fn_gb_index),
        len(fn_all_index)
    ]
})

display(overlap_summary)

,comparison,shared_false_negatives
0,Logistic Regression ∩ Random Forest,7
1,Logistic Regression ∩ Gradient Boosting,5
2,Random Forest ∩ Gradient Boosting,9
3,All three models,5


In [26]:
debug_df["missed_by_log_reg"] = (
    (debug_df["y_true"] == 1) &
    (debug_df["log_reg_pred"] == 0)
)

debug_df["missed_by_random_forest"] = (
    (debug_df["y_true"] == 1) &
    (debug_df["random_forest_pred"] == 0)
)

debug_df["missed_by_gradient_boosting"] = (
    (debug_df["y_true"] == 1) &
    (debug_df["gradient_boosting_pred"] == 0)
)

debug_df["missed_by_n_models"] = (
    debug_df["missed_by_log_reg"].astype(int) +
    debug_df["missed_by_random_forest"].astype(int) +
    debug_df["missed_by_gradient_boosting"].astype(int)
)

In [27]:
missed_count_summary = (
    debug_df[debug_df["y_true"] == 1]["missed_by_n_models"]
    .value_counts()
    .sort_index()
)

display(missed_count_summary)

missed_by_n_models
0    47
1     8
2     6
3     5
Name: count, dtype: int64

In [28]:
hard_cases = debug_df[
    (debug_df["y_true"] == 1) &
    (debug_df["missed_by_n_models"] > 0)
].copy()
diagnostic_cols = [
    "y_true",
    "log_reg_pred",
    "log_reg_proba",
    "random_forest_pred",
    "random_forest_proba",
    "gradient_boosting_pred",
    "gradient_boosting_proba",
    "Process temperature [K]",
    "Temperature difference",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Power [W]",
    "Tool wear [min]",
    "Type_L",
    "Type_M"
]

display(
    hard_cases[diagnostic_cols + ["missed_by_n_models"]]
    .sort_values("missed_by_n_models", ascending=False)
)

,y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],Type_L,Type_M,missed_by_n_models
8199,1,0,0.070138,0,0.150,0,0.138585,310.7,11.5,1737,27.0,4911.25,225,1,0,3
8609,1,0,0.249070,0,0.110,0,0.057842,308.3,10.9,1475,40.5,6255.70,222,1,0,3
4034,1,0,0.431925,0,0.150,0,0.053293,310.8,8.8,1615,29.0,4904.55,235,1,0,3
9018,1,0,0.132736,0,0.035,0,0.053958,308.1,10.8,1615,35.4,5986.93,217,1,0,3
7510,1,0,0.137707,0,0.125,0,0.045325,311.8,11.3,1524,38.9,6208.16,214,1,0,3
9758,1,1,0.778214,0,0.080,0,0.091027,309.8,11.2,2271,16.2,3852.66,218,1,0,2
4475,1,0,0.163843,0,0.415,1,0.965152,310.5,7.8,1351,41.8,5913.71,10,1,0,2
8026,1,1,0.635897,0,0.320,0,0.319052,311.9,11.2,1399,50.2,7354.45,222,1,0,2
9663,1,1,0.590242,0,0.345,0,0.126378,310.1,11.0,1435,48.8,7333.32,229,1,0,2
2941,1,1,0.605943,0,0.070,0,0.015848,309.6,8.9,1996,19.8,4138.61,203,0,1,2


index: {4151, 4833, 4475}

In [29]:
df_raw = pd.read_csv('../data/raw/produkcja.csv')
df_raw['Power [W]'] = df_raw['Rotational speed [rpm]'] * df_raw['Torque [Nm]'] * (2 * np.pi / 60)
df_raw['Temperature difference'] = df_raw['Process temperature [K]'] - df_raw['Air temperature [K]']
df_raw['Kryterium_OSF'] = df_raw['Tool wear [min]'] * df_raw['Torque [Nm]']

In [30]:
fn_every = [3787, 442, 3760, 3829, 1085, 2494, 9653, 4151, 4833, 2941, 9663, 8026, 4475, 9758, 7510, 9018, 4034, 8609, 8199]

In [31]:
suspect_cases = df_raw.loc[fn_every].copy()
display(suspect_cases)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,Power [W],Temperature difference,Kryterium_OSF
3787,3788,L50967,L,302.3,310.8,1377,47.3,22,1,0,1,0,0,0,6820.617562,8.5,1040.6
442,443,L47622,L,297.4,308.5,1399,61.5,61,1,0,0,1,0,0,9009.930651,11.1,3751.5
3760,3761,L50940,L,302.3,310.9,1377,46.8,166,1,0,1,0,0,0,6748.518011,8.6,7768.8
3829,3830,H33243,H,302.3,310.9,1366,48.4,130,1,0,1,0,0,0,6923.483778,8.6,6292.0
1085,1086,L48265,L,297.0,307.8,1385,56.4,202,1,0,0,0,1,0,8180.078951,10.8,11392.8
2494,2495,L49674,L,299.1,308.8,1329,53.6,207,1,0,0,0,1,0,7459.648924,9.7,11095.2
9653,9654,L56833,L,299.0,309.9,1373,55.7,201,1,0,0,0,1,0,8008.558465,10.9,11195.7
4151,4152,M19011,M,301.9,310.4,1373,48.0,73,1,0,1,0,0,0,6901.450741,8.5,3504.0
4833,4834,L52013,L,303.4,311.9,1377,41.6,34,1,0,1,0,0,0,5998.682676,8.5,1414.4
2941,2942,M17801,M,300.7,309.6,1996,19.8,203,1,1,0,0,0,0,4138.608498,8.9,4019.4


442 - bardzo blisko dolnego krańca przedziału dla [PWF]

3760, 3787, 3829, 4151, 4833, 4475 - bardzo blisko kranców przedziału dla obu parametrów - rotational speed i temp diff [HDF]

1085, 2494, 9653, 9663, 8026 - algorytmy nie wiedziały o Kryterium_OSF [OSF]

2941, 9758 - wysoki Rotational speed oraz Tool wear [TWF] - w EDA nie znaleziono zależności między wysokim Rotational speed a TWF

7510, 9018, 4034, 8609, 8199 - wysoki Tool wear [TWF]


3787, 442, 3760, 3829, 1085, 2494, 9653, 4151 - jeden model pomylił się

4833, 2941, 9663, 8026, 4475, 9758 - dwa modele pomyliły się

7510, 9018, 4034, 8609, 8199 - wszystkie się pomyliły

In [32]:
mechanism_groups = {
    "PWF_boundary_case": [442],
    
    "HDF_boundary_case": [3760, 3787, 3829, 4151, 4833, 4475],
    
    "OSF_missing_engineered_criterion": [1085, 2494, 9653, 9663, 8026],
    
    "TWF_high_rotational_speed_and_tool_wear": [2941, 9758],
    
    "TWF_high_tool_wear": [7510, 9018, 4034, 8609, 8199]
}

In [36]:
print('debug_df:', debug_df.shape)
print("Random Forest:", type(random_forest).__name__)
print("Gradient Boosting:", type(gradient_boosting).__name__)

debug_df: (2000, 19)
Random Forest: RandomForestClassifier
Gradient Boosting: GradientBoostingClassifier


## Analiza fałszywych awarii - False Positives

In [37]:
fp_rf = debug_df[(debug_df['y_true']==0) & (debug_df['random_forest_pred']== 1)].copy()

fp_gb = debug_df[(debug_df['y_true']==0) & (debug_df['gradient_boosting_pred']== 1)].copy()

print("False positives - Random Forest:", len(fp_rf))
print("False positives - Gradient Boosting:", len(fp_gb))

False positives - Random Forest: 9
False positives - Gradient Boosting: 7


In [38]:
display(fp_rf)
display(fp_gb)


,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba,missed_by_log_reg,missed_by_random_forest,missed_by_gradient_boosting,missed_by_n_models
3764,1,0,311.0,8.6,1329,54.6,7598.82,176,0,1,0.964519,1,0.670,1,0.543277,False,False,False,0
4898,0,0,312.4,8.8,1460,53.7,8210.24,214,0,1,0.907559,1,0.610,0,0.391752,False,False,False,0
8198,0,1,310.7,11.4,1374,48.9,7035.97,222,0,1,0.526683,1,0.560,0,0.481971,False,False,False,0
7081,1,0,310.4,9.7,1366,52.6,7524.28,205,0,1,0.898386,1,0.640,0,0.248132,False,False,False,0
7677,1,0,311.7,11.1,1428,52.6,7865.79,208,0,1,0.640267,1,0.645,1,0.854691,False,False,False,0
4115,0,0,310.6,8.6,1346,54.7,7710.12,196,0,1,0.960953,1,0.580,0,0.460964,False,False,False,0
4110,1,0,310.6,8.6,1361,46.7,6655.85,182,0,1,0.840622,1,0.750,0,0.227386,False,False,False,0
7255,0,1,310.3,10.1,1704,29.4,5246.21,221,0,0,0.163921,1,0.530,0,0.063622,False,False,False,0
4862,1,0,312.1,8.6,1326,55.7,7734.41,114,0,1,0.920796,1,0.665,1,0.547624,False,False,False,0


,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba,missed_by_log_reg,missed_by_random_forest,missed_by_gradient_boosting,missed_by_n_models
6925,0,1,311.6,10.5,1266,55.5,7357.92,210,0,1,0.944372,0,0.195,1,0.742128,False,False,False,0
3764,1,0,311.0,8.6,1329,54.6,7598.82,176,0,1,0.964519,1,0.670,1,0.543277,False,False,False,0
7677,1,0,311.7,11.1,1428,52.6,7865.79,208,0,1,0.640267,1,0.645,1,0.854691,False,False,False,0
1507,0,1,308.6,10.6,1376,55.7,8026.06,214,0,1,0.860949,0,0.315,1,0.850336,False,False,False,0
7935,0,1,311.9,11.1,1356,44.3,6290.60,214,0,0,0.375464,0,0.275,1,0.589482,False,False,False,0
4862,1,0,312.1,8.6,1326,55.7,7734.41,114,0,1,0.920796,1,0.665,1,0.547624,False,False,False,0
9671,1,0,310.2,11.2,1412,44.1,6520.82,246,0,0,0.465850,0,0.130,1,0.656823,False,False,False,0


In [39]:
print("RF — wartości y_true:")
print(fp_rf["y_true"].value_counts())

print("\nRF — wartości predykcji:")
print(fp_rf["random_forest_pred"].value_counts())

print("\nGB — wartości y_true:")
print(fp_gb["y_true"].value_counts())

print("\nGB — wartości predykcji:")
print(fp_gb["gradient_boosting_pred"].value_counts())

RF — wartości y_true:
y_true
0    9
Name: count, dtype: int64

RF — wartości predykcji:
random_forest_pred
1    9
Name: count, dtype: int64

GB — wartości y_true:
y_true
0    7
Name: count, dtype: int64

GB — wartości predykcji:
gradient_boosting_pred
1    7
Name: count, dtype: int64


In [42]:
fp_rf_summary = fp_rf.drop(columns=["log_reg_pred", "log_reg_proba", "gradient_boosting_pred", "gradient_boosting_proba", "missed_by_log_reg", "missed_by_gradient_boosting", "missed_by_n_models", "missed_by_random_forest"]).copy()
display(fp_rf_summary)

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,random_forest_pred,random_forest_proba
3764,1,0,311.0,8.6,1329,54.6,7598.82,176,0,1,0.670
4898,0,0,312.4,8.8,1460,53.7,8210.24,214,0,1,0.610
8198,0,1,310.7,11.4,1374,48.9,7035.97,222,0,1,0.560
7081,1,0,310.4,9.7,1366,52.6,7524.28,205,0,1,0.640
7677,1,0,311.7,11.1,1428,52.6,7865.79,208,0,1,0.645
4115,0,0,310.6,8.6,1346,54.7,7710.12,196,0,1,0.580
4110,1,0,310.6,8.6,1361,46.7,6655.85,182,0,1,0.750
7255,0,1,310.3,10.1,1704,29.4,5246.21,221,0,1,0.530
4862,1,0,312.1,8.6,1326,55.7,7734.41,114,0,1,0.665


False positives dla Random Forest: 3764
4898
8198
7081
7677
4115
4110
7255
4862

In [56]:
display(fp_rf_summary["random_forest_proba"].min())
display(fp_rf_summary["random_forest_proba"].max())

np.float64(0.53)

np.float64(0.75)

Przedział Random Forest Proba = [0.53; 0.75]

In [43]:
fp_gb_summary = fp_gb.drop(columns=["log_reg_pred", "log_reg_proba", "random_forest_pred", "random_forest_proba", "missed_by_log_reg", "missed_by_gradient_boosting", "missed_by_n_models", "missed_by_random_forest"]).copy()
display(fp_gb_summary)

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,gradient_boosting_pred,gradient_boosting_proba
6925,0,1,311.6,10.5,1266,55.5,7357.92,210,0,1,0.742128
3764,1,0,311.0,8.6,1329,54.6,7598.82,176,0,1,0.543277
7677,1,0,311.7,11.1,1428,52.6,7865.79,208,0,1,0.854691
1507,0,1,308.6,10.6,1376,55.7,8026.06,214,0,1,0.850336
7935,0,1,311.9,11.1,1356,44.3,6290.60,214,0,1,0.589482
4862,1,0,312.1,8.6,1326,55.7,7734.41,114,0,1,0.547624
9671,1,0,310.2,11.2,1412,44.1,6520.82,246,0,1,0.656823


False positives dla Gradient Boosting: 6925 3764 7677 1507 7935 4862 9671

In [59]:
display(fp_gb_summary["gradient_boosting_proba"].min().round(2))
display(fp_gb_summary["gradient_boosting_proba"].max().round(2))

np.float64(0.54)

np.float64(0.85)

Przedział Gradient Boosting Proba = (0.54; 0.85)

In [46]:
common_fp = fp_rf_summary.merge(fp_gb_summary, left_index=True, right_index=True, suffixes=('_rf', '_gb'))
display(common_fp)

,Type_L_rf,Type_M_rf,Process temperature [K]_rf,Temperature difference_rf,Rotational speed [rpm]_rf,Torque [Nm]_rf,Power [W]_rf,Tool wear [min]_rf,y_true_rf,random_forest_pred,...,Type_M_gb,Process temperature [K]_gb,Temperature difference_gb,Rotational speed [rpm]_gb,Torque [Nm]_gb,Power [W]_gb,Tool wear [min]_gb,y_true_gb,gradient_boosting_pred,gradient_boosting_proba
3764,1,0,311.0,8.6,1329,54.6,7598.82,176,0,1,...,0,311.0,8.6,1329,54.6,7598.82,176,0,1,0.543277
7677,1,0,311.7,11.1,1428,52.6,7865.79,208,0,1,...,0,311.7,11.1,1428,52.6,7865.79,208,0,1,0.854691
4862,1,0,312.1,8.6,1326,55.7,7734.41,114,0,1,...,0,312.1,8.6,1326,55.7,7734.41,114,0,1,0.547624
